# Movie Recommendation System

In [ ]:
import random
import pandas as pd
import numpy as np
from copy import deepcopy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
users = pd.read_csv('data/user_profiles.csv')
movies = pd.read_csv('data/movie_profiles.csv')
ratings = pd.read_csv('data/ml-latest-small/ratings.csv')

In [3]:
users.head()

,userId,avg_rating,rating_std,num_ratings,min_rating,max_rating,first_rating_time,last_rating_time,rating_range,rating_consistency,...,count_Romance,count_Sci-Fi,count_Thriller,count_War,count_Western,num_tags,tag_diversity,high_ratings_ratio,low_ratings_ratio,extreme_ratings_ratio
0,1,4.366379,0.800048,232,1.0,5.0,964980499,965719662,4.0,1.111052,...,26.0,40.0,55.0,22.0,7.0,0.0,0.0,0.862069,0.025862,0.538793
1,2,3.948276,0.805615,29,2.0,5.0,1445714835,1445715340,3.0,1.104223,...,1.0,4.0,10.0,1.0,1.0,9.0,9.0,0.655172,0.034483,0.206897
2,3,2.435897,2.090642,39,0.5,5.0,1306463323,1306464293,4.5,0.456487,...,5.0,15.0,7.0,5.0,0.0,0.0,0.0,0.410256,0.538462,0.256410
3,4,3.555556,1.314204,216,1.0,5.0,945078428,1007574542,4.0,0.707112,...,58.0,12.0,38.0,7.0,10.0,0.0,0.0,0.592593,0.226852,0.402778
4,5,3.636364,0.990441,44,1.0,5.0,847434747,847435337,4.0,0.917061,...,11.0,2.0,9.0,3.0,2.0,0.0,0.0,0.522727,0.090909,0.250000


In [4]:
movies.head()

,movieId,title,avg_rating,rating_std,num_ratings,min_rating,max_rating,bayesian_avg,rating_confidence,rating_range,...,num_unique_tags,num_users_tagged,tag_diversity,has_tag_in_netflix_queue,has_tag_atmospheric,has_tag_sci-fi,has_tag_funny,has_tag_dark_comedy,num_unique_users,engagement_rate
0,1,Toy Story (1995),3.920930,0.834859,215.0,0.5,5.0,3.89,1.000000,4.5,...,2.0,3.0,0.666667,0.0,0.0,0.0,0.0,0.0,215,0.352459
1,2,Jumanji (1995),3.431818,0.881713,110.0,0.5,5.0,3.42,1.000000,4.5,...,4.0,2.0,1.000000,0.0,0.0,0.0,0.0,0.0,110,0.180328
2,3,Grumpier Old Men (1995),3.259615,1.054823,52.0,0.5,5.0,3.26,1.000000,4.5,...,2.0,1.0,1.000000,0.0,0.0,0.0,0.0,0.0,52,0.085246
3,4,Waiting to Exhale (1995),2.357143,0.852168,7.0,1.0,3.0,2.90,0.675037,2.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,7,0.011475
4,5,Father of the Bride Part II (1995),3.071429,0.907148,49.0,0.5,5.0,3.10,1.000000,4.5,...,2.0,1.0,1.000000,0.0,0.0,0.0,0.0,0.0,49,0.080328


In [5]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [6]:
training_data = ratings[['userId', 'movieId', 'rating']].copy()

training_data = training_data.merge(
    users, on='userId', how='left'
).merge(
    movies.drop('title', axis=1), on='movieId', how='left'
)

print(f'Columns: {training_data.columns}')
training_data.head()

Columns: Index(['userId', 'movieId', 'rating', 'avg_rating_x', 'rating_std_x',
       'num_ratings_x', 'min_rating_x', 'max_rating_x', 'first_rating_time',
       'last_rating_time',
       ...
       'num_unique_tags', 'num_users_tagged', 'tag_diversity_y',
       'has_tag_in_netflix_queue', 'has_tag_atmospheric', 'has_tag_sci-fi',
       'has_tag_funny', 'has_tag_dark_comedy', 'num_unique_users',
       'engagement_rate'],
      dtype='object', length=107)


,userId,movieId,rating,avg_rating_x,rating_std_x,num_ratings_x,min_rating_x,max_rating_x,first_rating_time,last_rating_time,...,num_unique_tags,num_users_tagged,tag_diversity_y,has_tag_in_netflix_queue,has_tag_atmospheric,has_tag_sci-fi,has_tag_funny,has_tag_dark_comedy,num_unique_users,engagement_rate
0,1,1,4.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,2.0,3.0,0.666667,0.0,0.0,0.0,0.0,0.0,215,0.352459
1,1,3,4.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,2.0,1.0,1.000000,0.0,0.0,0.0,0.0,0.0,52,0.085246
2,1,6,4.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,102,0.167213
3,1,47,5.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,3.0,2.0,1.000000,0.0,0.0,0.0,0.0,0.0,203,0.332787
4,1,50,5.0,4.366379,0.800048,232,1.0,5.0,964980499,965719662,...,6.0,2.0,1.000000,0.0,0.0,0.0,0.0,0.0,204,0.334426


In [7]:
X = training_data.drop(['userId', 'movieId', 'rating'], axis=1)
y = training_data['rating']

user_ids = training_data['userId']
movie_ids = training_data['movieId']

# Split: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f'Train: {X_train.shape}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')

Train: (70585, 104)
Val:   (15125, 104)
Test:  (15126, 104)


In [8]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [9]:
print(f'-- Standardized X_train:\n {X_train}')
print(f'\n-- Standardized X_val:\n {X_val}')
print(f'\n-- Standardized X_test:\n {X_test}')

-- Standardized X_train:
 [[ 0.66784609 -0.3591107  -0.41216397 ... -0.10310424 -0.0311001
  -0.0311001 ]
 [ 0.3353629   0.37875235 -0.82125541 ... -0.10310424  0.80628342
   0.80628342]
 [ 1.96986687 -1.98331243 -0.68642377 ... -0.10310424  1.36990694
   1.36990694]
 ...
 [-0.49299539  0.64888275  0.35698921 ... -0.10310424 -0.75575892
  -0.75575892]
 [-0.0186587  -0.30156955 -0.44280753 ... -0.10310424 -0.91679422
  -0.91679422]
 [ 0.87902184 -0.59834718 -0.34628033 ... -0.10310424  0.24265989
   0.24265989]]

-- Standardized X_val:
 [[ 1.41973463 -0.93679307 -0.82585194 ... -0.10310424 -0.73965539
  -0.73965539]
 [ 1.33310684 -1.43687961  0.18232096 ... -0.10310424 -0.01499658
  -0.01499658]
 [ 0.01381706  1.00927048 -0.47804761 ... -0.10310424 -0.93289775
  -0.93289775]
 ...
 [-0.27497756  0.21632738  0.46117729 ... -0.10310424 -0.88458716
  -0.88458716]
 [-0.63347098 -0.45395658 -0.58530005 ... -0.10310424 -0.83627657
  -0.83627657]
 [ 1.34664626  0.60370727 -0.68642377 ... -0.103

In [11]:
class Individual:
    def __init__(self, input_dim, chromosome=None):
        self.input_dim = input_dim

        if chromosome is None:
            self.chromosome = self.minimal_chromosome()
        else:
            self.chromosome = chromosome

        self.model = None
        self.fitness = None
        self.val_loss = None
        self.complexity = self.calculate_complexity()

    # Start with a minimal network (zero hidden layers), complexity will evolve eventually
    @staticmethod
    def minimal_chromosome():
        return {
            'num_layers': 0,
            'layer_sizes': [],
            'activations': [],
            'dropout_rates': [],
            'learning_rate': 0.001,
            'batch_size': 128,
        }

    # Calculate network complexity for fitness penalty
    def calculate_complexity(self):
        # Input to first layer (or output if no hidden)
        if self.chromosome['num_layers'] == 0:
            num_params = self.input_dim * 1  # Direct to output
        else:
            num_params = self.input_dim * self.chromosome['layer_sizes'][0]

            # Hidden layers
            for i in range(self.chromosome['num_layers'] - 1):
                num_params += self.chromosome['layer_sizes'][i] * self.chromosome['layer_sizes'][i+1]

            # Last hidden to output
            num_params += self.chromosome['layer_sizes'][-1] * 1

        return num_params

    # Build a neural network from chromosome
    def build_model(self):
        model = keras.Sequential()

        # Input layer
        model.add(layers.Input(shape=(self.input_dim,)))

        # Hidden layers (if any)
        for i in range(self.chromosome['num_layers']):
            model.add(layers.Dense(
                self.chromosome['layer_sizes'][i],
                activation=self.chromosome['activations'][i],
                kernel_regularizer=keras.regularizers.l2(0.001)
            ))
            model.add(layers.Dropout(self.chromosome['dropout_rates'][i]))

        # Output layer
        model.add(layers.Dense(1, activation='linear'))

        # Compile
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self.chromosome['learning_rate']),
            loss='mse',
            metrics=['mae']
        )

        self.model = model
        return model

    def calc_fitness(self, x_train, y_train, x_val, y_val, epochs=10, complexity_penalty=0.00001):
        if self.model is None:
            self.build_model()

        # Early stopping
        early_stop = keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=3,
            restore_best_weights=True
        )

        # Train
        history = self.model.fit(
            x_train, y_train,
            validation_data=(x_val, y_val),
            epochs=epochs,
            batch_size=self.chromosome['batch_size'],
            callbacks=[early_stop],
            verbose=0
        )

        # Get the best validation loss
        val_loss = min(history.history['val_loss'])

        # Penalize complexity (prefer simpler networks)
        self.complexity = self.calculate_complexity()
        complexity_cost = complexity_penalty * self.complexity

        # Fitness = performance - complexity
        self.fitness = -(val_loss + complexity_cost)
        self.val_loss = val_loss

        return self.fitness

    def __lt__(self, other):
        return self.fitness < other.fitness

    def __repr__(self):
        return f"NEAT(fit={self.fitness:.4f}, loss={self.val_loss:.4f}, layers={self.chromosome['num_layers']}, params={self.complexity})"

In [12]:
class GeneticAlgorithm:
    def __init__(
        self,
        population_size: int,
        num_generations: int,
        mutation_prob: float,
        add_node_prob: float,  # Probability of adding a layer
        elitism_size: float,
        selection_type: str = 'tournament',
        tournament_size: int = 3,
        training_epochs: int = 10,
        complexity_penalty: float = 0.00001,
    ):
        self.population_size = population_size
        self.num_elite = int(population_size * elitism_size)
        if self.num_elite % 2 != self.population_size % 2:
            self.num_elite += 1
        self.mutation_prob = mutation_prob
        self.add_node_prob = add_node_prob  # NEAT: structural mutation
        self.selection_type = selection_type
        self.tournament_size = tournament_size
        self.num_generations = num_generations
        self.training_epochs = training_epochs
        self.complexity_penalty = complexity_penalty

        self.history = {
            'best_fitness': [],
            'avg_complexity': [],
            'avg_layers': [],
        }

    def selection(self, population):
        if self.selection_type == 'tournament':
            participants = random.sample(population, self.tournament_size)
            return max(participants, key=lambda x: x.fitness)
        else:
            raise ValueError(f'unknown selection_type: {self.selection_type}')

    @staticmethod
    def crossover(parent1, parent2):

        fitter = parent1 if parent1.fitness > parent2.fitness else parent2
        weaker = parent2 if parent1.fitness > parent2.fitness else parent1

        # Start with fitter parent's structure
        child_chromo = deepcopy(fitter.chromosome)

        # Randomly inherit some traits from weaker parent
        if random.random() < 0.3:
            child_chromo['learning_rate'] = weaker.chromosome['learning_rate']

        if random.random() < 0.3:
            child_chromo['batch_size'] = weaker.chromosome['batch_size']

        # For layers that exist in both, maybe swap
        min_layers = min(len(parent1.chromosome['layer_sizes']),
                        len(parent2.chromosome['layer_sizes']))

        for i in range(min_layers):
            if random.random() < 0.3:
                child_chromo['layer_sizes'][i] = weaker.chromosome['layer_sizes'][i]
                child_chromo['activations'][i] = weaker.chromosome['activations'][i]
                child_chromo['dropout_rates'][i] = weaker.chromosome['dropout_rates'][i]

        return child_chromo

    # Add or remove layers
    def structural_mutation(self, chromosome):
        mutated = deepcopy(chromosome)

        if random.random() < self.add_node_prob:
            if mutated['num_layers'] < 5:  # Max 5 hidden layers
                mutated['num_layers'] += 1
                # Add new layer with modest size
                new_size = random.choice([64, 128])
                mutated['layer_sizes'].append(new_size)
                mutated['activations'].append('relu')
                mutated['dropout_rates'].append(random.uniform(0.2, 0.4))
                print(f"\tAdded layer! Now {mutated['num_layers']} layers", flush=True)

        elif random.random() < self.add_node_prob * 0.3:  # 30% chance compared to adding
            if mutated['num_layers'] > 0:
                mutated['num_layers'] -= 1
                mutated['layer_sizes'].pop()
                mutated['activations'].pop()
                mutated['dropout_rates'].pop()
                print(f"\tRemoved layer! Now {mutated['num_layers']} layers", flush=True)

        return mutated

    # Regular mutation
    def parameter_mutation(self, chromosome):
        mutated = deepcopy(chromosome)

        # Mutate learning rate
        if random.random() < self.mutation_prob:
            mutated['learning_rate'] = random.choice([0.0005, 0.001, 0.005])

        # Mutate batch size
        if random.random() < self.mutation_prob:
            mutated['batch_size'] = random.choice([128, 256])

        # Mutate existing layers
        for i in range(len(mutated['layer_sizes'])):
            if random.random() < self.mutation_prob:
                mutated['layer_sizes'][i] = random.choice([64, 128, 256])

            if random.random() < self.mutation_prob:
                mutated['activations'][i] = random.choice(['relu', 'tanh'])

            if random.random() < self.mutation_prob:
                mutated['dropout_rates'][i] = random.uniform(0.2, 0.4)

        return mutated

    def solve(self, x_train, y_train, x_val, y_val):
        input_dim = x_train.shape[1]

        print(f"--- Starting evolution...")
        print(f"--- Populations: {self.population_size}, Generations: {self.num_generations}")
        print(f"--- Starting with MINIMAL topology (0 hidden layers)\n")

        population = [Individual(input_dim) for _ in range(self.population_size)]

        # Evaluate initial population
        print("--- Evaluating minimal networks...")
        for i, ind in enumerate(population):
            ind.calc_fitness(x_train, y_train, x_val, y_val,
                           epochs=self.training_epochs,
                           complexity_penalty=self.complexity_penalty)
            print(f"\t[{i+1}/{self.population_size}] {ind}")

        best_overall = max(population, key=lambda x: x.fitness)

        # Evolution
        for gen in range(self.num_generations):
            print(f"\n{'='*70}")
            print(f"--- Generation {gen+1}/{self.num_generations}")
            print(f"{'='*70}")

            population.sort(reverse=True, key=lambda x: x.fitness)

            # Statistics
            avg_layers = np.mean([ind.chromosome['num_layers'] for ind in population])
            avg_complexity = np.mean([ind.complexity for ind in population])

            self.history['avg_layers'].append(avg_layers)
            self.history['avg_complexity'].append(avg_complexity)

            new_population = population[:self.num_elite]

            print(f"--- Top 3:")
            for i, ind in enumerate(population[:3]):
                print(f"  {i+1}. {ind}")

            print(f"\n--- Population Stats:")
            print(f"\tAvg Layers: {avg_layers:.2f}")
            print(f"\tAvg Params: {avg_complexity:.0f}")
            print(f"\tComplexity Range: {min(ind.complexity for ind in population)} - {max(ind.complexity for ind in population)}")

            # Generate children
            print(f"\n--- Generating children...")
            for i in range(self.num_elite, self.population_size):
                parent1 = self.selection(population)
                parent2 = self.selection(population)

                child_chromo = self.crossover(parent1, parent2)
                child_chromo = self.structural_mutation(child_chromo)
                child_chromo = self.parameter_mutation(child_chromo)

                child = Individual(input_dim, child_chromo)
                child.calc_fitness(x_train, y_train, x_val, y_val,
                                 epochs=self.training_epochs,
                                 complexity_penalty=self.complexity_penalty)

                print(f"\tChild {i-self.num_elite+1}: {child}")
                new_population.append(child)

            population = new_population

            current_best = max(population, key=lambda x: x.fitness)
            if current_best.fitness > best_overall.fitness:
                best_overall = deepcopy(current_best)
                print(f"\n\tNEW BEST: {best_overall}")

        return best_overall

In [13]:
# Use a sample for faster evolution
SAMPLE_SIZE = 10000
indices = np.random.choice(len(X_train), SAMPLE_SIZE, replace=False)
X_train_sample = X_train[indices]
y_train_sample = y_train.iloc[indices]

val_indices = np.random.choice(len(X_val), 2000, replace=False)
X_val_sample = X_val[val_indices]
y_val_sample = y_val.iloc[val_indices]

ga = GeneticAlgorithm(
    population_size=15,
    num_generations=12,
    mutation_prob=0.2,
    add_node_prob=0.3,          # 30% chance to add/remove layer
    elitism_size=0.2,
    selection_type='tournament',
    tournament_size=3,
    training_epochs=8,
    complexity_penalty=0.00001,  # Penalize complex networks
)

best = ga.solve(X_train_sample, y_train_sample, X_val_sample, y_val_sample)

print("\n" + "="*70)
print("--- NEAT EVOLUTION COMPLETE ---")
print("="*70)
print(f"Best: {best}")
print(f"Architecture evolved: {best.chromosome}")

--- Starting evolution...
--- Populations: 15, Generations: 12
--- Starting with MINIMAL topology (0 hidden layers)

--- Evaluating minimal networks...
	[1/15] NEAT(fit=-9.1269, loss=9.1259, layers=0, params=104)
	[2/15] NEAT(fit=-9.1890, loss=9.1880, layers=0, params=104)
	[3/15] NEAT(fit=-9.1346, loss=9.1336, layers=0, params=104)
	[4/15] NEAT(fit=-9.1834, loss=9.1824, layers=0, params=104)
	[5/15] NEAT(fit=-9.0802, loss=9.0792, layers=0, params=104)
	[6/15] NEAT(fit=-9.1250, loss=9.1239, layers=0, params=104)
	[7/15] NEAT(fit=-9.1043, loss=9.1032, layers=0, params=104)
	[8/15] NEAT(fit=-9.1445, loss=9.1435, layers=0, params=104)
	[9/15] NEAT(fit=-9.0506, loss=9.0496, layers=0, params=104)
	[10/15] NEAT(fit=-9.1446, loss=9.1436, layers=0, params=104)
	[11/15] NEAT(fit=-9.1060, loss=9.1049, layers=0, params=104)
	[12/15] NEAT(fit=-9.0898, loss=9.0887, layers=0, params=104)
	[13/15] NEAT(fit=-9.0546, loss=9.0536, layers=0, params=104)
	[14/15] NEAT(fit=-9.0835, loss=9.0825, layers=0, p

In [18]:
# Train on full data
print("\n--- Retraining the best model on FULL dataset:\n")

best.model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=best.chromosome['batch_size'],
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

# Model evaluation
test_loss, test_mae = best.model.evaluate(X_test, y_test)

print(f"\n--- Final results:")
print(f"\tTest loss: {test_loss:.4f}")
print(f"\tTest MAE: {test_mae:.4f}")
print(f"\tFinal complexity: {best.complexity} parameters")


--- Retraining the best model on FULL dataset:

Epoch 1/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6636 - mae: 0.6111 - val_loss: 0.6379 - val_mae: 0.5896
Epoch 2/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6602 - mae: 0.6086 - val_loss: 0.6436 - val_mae: 0.5972
Epoch 3/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6643 - mae: 0.6107 - val_loss: 0.6510 - val_mae: 0.6040
Epoch 4/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6580 - mae: 0.6085 - val_loss: 0.6438 - val_mae: 0.5935
Epoch 5/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6585 - mae: 0.6086 - val_loss: 0.6398 - val_mae: 0.5927
Epoch 6/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.6610 - mae: 0.6095 - val_loss: 0.6452 - val_mae: 0.5926
473/473 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.6421 - mae: 0.5942

--- Final results:
	Test loss: 0.6341
	Test MAE: 0.5924
	Final complexity: 6720 parameters
